## README:

This notebook contains code to download data from external databases using APIs.

Pre-requisites:
1) FRED API key value to be set to the FRED_API_KEY environment variable.

**NOTE**: Cells labeled with '## ----- CONFIG ----- ##' contain parameters that need to be set manually 

In [1]:
import io
import os
import pandas as pd

from contextlib import redirect_stdout
from dotenv import load_dotenv

## Global config

In [2]:
## ----- CONFIG ----- ##
WRITE_MODE = 'w' # 'x': do not overwrite

## Load env variables

In [3]:
load_dotenv(r'../.env')

True

In [4]:
DATA_DIR = os.getenv('DATA_DIR')

# create data directory if does not exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'FRED'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'YFINANCE'), exist_ok=True)

## yfinance - Stock prices

In [6]:
import yfinance as yf

### Configuration

In [7]:
## ----- CONFIG ----- ##
YF_VERSION = '001' # set the version of data

# refer to parameters on https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html
YF_PARAMS = {
    'period': 'max',
    'interval': '1d',
    'start': '1975-01-01',
    'end': '2025-12-10',
    'keepna': True, # keep NAs to process manually
    'actions': False,
    'multi_level_index': True
}

# stock tickers to get data for
TICKER_LIST = [
    'SPY', 
    'AAPL', 
    'NVDA', 
    'MSFT', 
    'AMZN', 
    'GOOG', 
    'JPM', 
    'XOM',
    'PG', 
    'UNH',
    'F'
]

### Download data

In [ ]:
# download data in bulk
df = yf.download(
    tickers=TICKER_LIST,
    **YF_PARAMS
)

df.shape

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  11 of 11 completed


(12843, 55)

In [9]:
# export data to local storage
# skips if already exists (prevents overwrite)
file_name = f"prices_{YF_VERSION}.csv"
out_path = os.path.join(DATA_DIR, 'YFINANCE', file_name)
try:
    df.reset_index().to_csv(out_path, index=False, mode=WRITE_MODE)
except FileExistsError:
    print(f"{file_name} already exists in the output directory.")

## FRED API - Economic data

In [10]:
import pyfredapi as pf
from pyfredapi._base import FredAPIRequestError 

API_KEY = os.getenv('FRED_API_KEY')

### Configuration

In [11]:
## ----- CONFIG ----- ##
VERSION = '001' # set the version of data

# earliest and latest data to download
# actual data may not be available for the entire span
FRED_PARAMS = {
    'observation_start': '1975-01-01',
    'observation_end': '2025-12-10'
}

# IDs of FRED data series to download
# search by categories on https://fred.stlouisfed.org/categories
SERIES_ID_LIST = [
    'VIXCLS',
    'DFF',
    'T10Y2Y',
    'REAINTRATREARAT10Y',
    'BAMLH0A0HYM2',
    'CPIAUCSL',
    'ICSA',
    'PPIACO',
    'USSTHPI',
    'BOGMBASE',
    'DTWEXBGS',
    'USREC',
    'USSLIND',
    'USEPUINDXD'
]

### Download data

In [12]:
# store printed string
buffer = io.StringIO()

# download data for each series
with redirect_stdout(buffer):
    for series_id in SERIES_ID_LIST:
        # get series info
        try:
            series_info = pf.get_series_info(series_id=series_id, api_key=API_KEY)
        except FredAPIRequestError as e:
            msg = str(e)
            if 'The series does not exist' in msg:
                print(f'The series "{series_id}" does not exist')
            else:
                print(msg)
            continue

        # print series metadata
        print(f'id: {series_id} | title: {series_info.title}')
        print(f'obs range: {series_info.observation_start} to {series_info.observation_end}')
        print(f'freq: {series_info.frequency} | units: {series_info.units}')
        print(series_info.seasonal_adjustment)

        # get data
        df = pf.get_series(series_id=series_id, api_key=API_KEY, **FRED_PARAMS)
        print(df.shape, '\n')

        # export data to local storage
        # skips if already exists (prevents overwrite)
        file_name = f'{series_id}_{VERSION}.csv'
        out_path = os.path.join(DATA_DIR, 'FRED', file_name)
        try:
            df.to_csv(out_path, index=False, mode=WRITE_MODE)
        except FileExistsError:
            print(f'{file_name} already exists in the output directory.')

In [ ]:
# write metadata
metadata_file = os.path.join(DATA_DIR, 'FRED', f"metadata_{VERSION}.txt")
with open(metadata_file, "w") as f:
    f.write(buffer.getvalue())